# Life Quality Index and SWTP

This tutorial follows the continuous steel-bar decision example in [Schubert and Faber (2009)](https://www.jcss-lc.org/publications/raie/11_example_jcss_ms_2.pdf). The cross-sectional area changes the resistance, FORM estimates the failure probability, and the JCSS LQI criterion is compared with the economic optimum.

The LQI helpers in Pystra are post-processing tools. They do not change the stochastic model or the reliability method; they use the `pf` and `beta` results returned by FORM, SORM, or simulation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pystra as ra

pd.options.display.float_format = "{:,.4g}".format

## Reliability model

The limit state is written as

$$g(f_y, A_s, S) = f_y A_s - 1000 S$$

where `A_s` is the design variable. For each value of `A_s`, FORM computes the failure probability and reliability index.

In [ ]:
def lsf(fy, As, S):
    return fy * As - 1000 * S


def run_reliability(As):
    limit_state = ra.LimitState(lsf)
    model = ra.StochasticModel()
    model.addVariable(ra.Lognormal("fy", 260, 18.2))
    model.addVariable(ra.Constant("As", As))
    model.addVariable(ra.Gumbel("S", 9.5, 1.5))

    options = ra.AnalysisOptions()
    options.setE1(1e-6)
    options.setE2(1e-6)

    form = ra.Form(
        analysis_options=options,
        limit_state=limit_state,
        stochastic_model=model,
    )
    form.run()
    return {"pf": float(np.atleast_1d(form.getFailure())[0]), "beta": form.getBeta()}

In [ ]:
as_values = np.array([70, 75, 80, 85.1, 90, 93, 95, 100], dtype=float)
study = ra.DesignStudy(variable="As", values=as_values, analysis=run_reliability)

reliability = study.evaluate()
reliability

## SWTP and the LQI target

Rackwitz's JCSS SWTP table is retained as a 1999 PPPUS$ anchor [Rackwitz (2008)](https://www.jcss-lc.org/publications/raie/06_risk_backgrounddoc_lqi_philosophy.pdf). For present-day studies, `indexed=True` returns an explicitly indexed value using World Bank GDP per capita PPP factors [World Bank WDI](https://data.worldbank.org/indicator/NY.GDP.PCAP.PP.CD). This keeps the literature value and the update method visible.

In [ ]:
swtp_anchor = ra.SWTP.from_country("CH")
swtp_indexed = ra.SWTP.from_country("CH", indexed=True)

pd.DataFrame(
    [
        {"basis": "Rackwitz anchor", "value_per_life": swtp_anchor.value_per_life, "price_year": swtp_anchor.price_year},
        {"basis": "GDP PPP indexed", "value_per_life": swtp_indexed.value_per_life, "price_year": swtp_indexed.price_year},
    ]
)

For the Fischer, Barnardo, and Faber target table [Fischer et al. (2012)](https://www.researchgate.net/publication/289533079_Deriving_target_reliabilities_from_the_LQI), define

$$K_1 = \frac{C_1}{\mathrm{SWTP} N_F}$$

where `C1` is the marginal safety cost and `N_F` is the expected number of fatalities conditional on failure. The result is a minimum LQI target reliability.

In [ ]:
expected_fatalities = 12
marginal_safety_cost = 5000

k1 = ra.lqi_k1(
    safety_cost_rate=marginal_safety_cost,
    swtp=swtp_indexed,
    expected_fatalities=expected_fatalities,
)
target = ra.lqi_target_reliability(k1)

pd.DataFrame([target.__dict__])

## Cost-benefit objective and LQI feasibility

The objective below follows the JCSS steel-bar calculation [Schubert and Faber (2009)](https://www.jcss-lc.org/publications/raie/11_example_jcss_ms_2.pdf) with a constant annual benefit, construction cost proportional to `A_s`, and failure costs discounted over the service life.

In [ ]:
costs = ra.CostBenefitModel(
    benefit_rate=1.2e4,
    interest_rate=0.02,
    service_life=100,
    construction_cost=lambda As: 5000 * As,
    failure_cost=lambda As: 5000 * As + 12 * 1.8e6 + 3e4,
)

assessment = ra.LQIAssessment(
    study=study,
    costs=costs,
    swtp=swtp_indexed,
    consequence=ra.FatalityConsequence(people_exposed=expected_fatalities),
    target=target,
)

results = assessment.evaluate()
results["construction_cost"] = 5000 * results["As"]
results["jcss_lqi_risk_cost"] = ra.jcss_lqi_risk_cost(
    results["construction_cost"],
    results["pf"],
    swtp_indexed,
    expected_fatalities,
)

results[[
    "As",
    "pf",
    "beta",
    "objective",
    "annualized_safety_cost",
    "target_pf",
    "lqi_acceptable",
    "jcss_lqi_risk_cost",
]]

In [ ]:
economic_best = results.loc[results["objective"].idxmax()]
feasible_best = results.loc[results[results["lqi_acceptable"]]["objective"].idxmax()]

pd.DataFrame(
    [
        {"selection": "economic optimum", "As": economic_best["As"], "pf": economic_best["pf"], "objective": economic_best["objective"], "lqi_acceptable": economic_best["lqi_acceptable"]},
        {"selection": "best LQI-feasible", "As": feasible_best["As"], "pf": feasible_best["pf"], "objective": feasible_best["objective"], "lqi_acceptable": feasible_best["lqi_acceptable"]},
    ]
)

In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(results["As"], results["objective"], marker="o", label="objective")
ax1.set_xlabel("Reinforcement area As")
ax1.set_ylabel("Objective")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.semilogy(results["As"], results["pf"], marker="s", color="tab:red", label="failure probability")
ax2.axhline(target.pf, color="tab:red", linestyle="--", label="LQI target")
ax2.set_ylabel("Failure probability")

lines = ax1.get_lines() + ax2.get_lines()
labels = [line.get_label() for line in lines]
ax1.legend(lines, labels, loc="best")
fig.tight_layout()

## JCSS paper 11 summary plot

The JCSS paper's Figure 1 combines the failure probability, annualized safety cost, marginal LQI condition, and objective for the continuous design. The cells below reproduce that summary from the stated distributions and Table 1 parameters. The probability curve uses the integral definition in Equation (1); the earlier table shows the same reliability problem through Pystra's FORM workflow.

In [ ]:
from scipy.integrate import quad
from scipy.interpolate import PchipInterpolator
from scipy.optimize import brentq, minimize_scalar
from scipy.stats import gumbel_r, lognorm


steel_cost_rate = 5000.0
benefit_rate = 1.2e4
compensation_cost = 1.8e6
cleaning_cost = 3.0e4
interest_rate = 0.02
service_life = 100.0
people_exposed = 12.0
probability_death_given_failure = 1.0
expected_fatalities_jcss = people_exposed * probability_death_given_failure

gdp_per_capita = 35931.0
demographic_constant = 18.9
mortality_rate = 0.175
paper_swtp = ra.SWTP.from_lqi(
    gross_domestic_product_per_capita=gdp_per_capita,
    mortality_rate=mortality_rate,
    demographic_constant=demographic_constant,
    currency="CHF",
    price_year=2009,
    source="Schubert and Faber (2009), Table 1",
)

yield_mean = 260.0
yield_std = 18.2
load_mean = 9.50
load_std = 1.50

yield_cov = yield_std / yield_mean
yield_sigma = np.sqrt(np.log1p(yield_cov**2))
yield_scale = yield_mean / np.sqrt(1.0 + yield_cov**2)
load_scale = load_std * np.sqrt(6.0) / np.pi
load_location = load_mean - np.euler_gamma * load_scale


def steel_bar_failure_probability(As):
    def integrand(load):
        threshold = 1000.0 * load / As
        return lognorm.cdf(threshold, s=yield_sigma, scale=yield_scale) * gumbel_r.pdf(
            load, loc=load_location, scale=load_scale
        )

    return quad(integrand, 0.0, np.inf, epsabs=1e-12, epsrel=1e-10, limit=100)[0]


jcss_as_grid = np.linspace(70.0, 100.0, 121)
jcss_pf_values = np.array([steel_bar_failure_probability(As) for As in jcss_as_grid])
jcss_pf = PchipInterpolator(jcss_as_grid, jcss_pf_values)

publication_costs = ra.CostBenefitModel(
    benefit_rate=benefit_rate,
    interest_rate=interest_rate,
    service_life=service_life,
    construction_cost=lambda As: steel_cost_rate * As,
    failure_cost=lambda As: steel_cost_rate * As
    + expected_fatalities_jcss * compensation_cost
    + cleaning_cost,
)


def pf_at(As):
    return float(jcss_pf(As))


def annualized_cost(As):
    return publication_costs.annualized_safety_cost(As, pf_at(As))


def objective_at(As):
    return publication_costs.objective(As, pf_at(As))


def paper_lqi_derivative(As):
    dcost = ra.finite_difference_derivative(annualized_cost, As, step=1e-3)
    drate = ra.finite_difference_derivative(pf_at, As, step=1e-3)
    margin = ra.jcss_lqi_acceptability_margin(
        dcost, drate, paper_swtp, expected_fatalities_jcss
    )
    return -margin / paper_swtp.for_lives(expected_fatalities_jcss)


computed_optimum = minimize_scalar(
    lambda As: -objective_at(As), bounds=(70.0, 100.0), method="bounded"
).x
computed_lqi_limit = brentq(paper_lqi_derivative, 80.0, 100.0)

# Rounded alternatives reported in Figure 1 of Schubert and Faber (2009).
optimal_as = 85.1
acceptable_as = 93.0

pd.DataFrame(
    [
        {
            "alternative": "economic optimum",
            "As": optimal_as,
            "computed_As": computed_optimum,
            "pf": pf_at(optimal_as),
            "annualized_safety_cost": annualized_cost(optimal_as),
            "objective": objective_at(optimal_as),
        },
        {
            "alternative": "LQI acceptable",
            "As": acceptable_as,
            "computed_As": computed_lqi_limit,
            "pf": pf_at(acceptable_as),
            "annualized_safety_cost": annualized_cost(acceptable_as),
            "objective": objective_at(acceptable_as),
        },
    ]
)


In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(8.5, 9.0), sharex=True)
fig.subplots_adjust(hspace=0.12)

opt_color = "#2f3aa6"
acc_color = "#d62728"
line_color = "#27349a"

for ax in axes:
    ax.set_facecolor("0.86")
    ax.grid(True, which="major", color="black", linestyle=(0, (4, 7)), linewidth=0.7)
    ax.set_xlim(70, 100)
    ax.axvline(optimal_as, color="black", linewidth=1.2)
    ax.axvline(acceptable_as, color="black", linewidth=1.2)

axes[0].semilogy(jcss_as_grid, jcss_pf_values, color=line_color, linewidth=1.8)
axes[0].plot(optimal_as, pf_at(optimal_as), marker="o", markersize=9, markerfacecolor="none", markeredgecolor=opt_color, markeredgewidth=1.8)
axes[0].plot(acceptable_as, pf_at(acceptable_as), marker="s", markersize=9, markerfacecolor="none", markeredgecolor=acc_color, markeredgewidth=1.8)
axes[0].set_ylim(1e-3, 1e-6)
axes[0].set_ylabel(r"$P_f(A)$ [yr$^{-1}$]")
axes[0].text(70.8, 1.7e-6, "A)", fontsize=14, fontstyle="italic")
axes[0].annotate(
    rf"$P_{{f,opt}}=2.57e-05$ [yr$^{{-1}}$]",
    xy=(optimal_as, pf_at(optimal_as)),
    xytext=(77.5, 1.7e-5),
    arrowprops={"arrowstyle": "-", "color": "0.2", "linewidth": 0.8},
)
axes[0].annotate(
    rf"$P_{{f,LQI}}=5.15e-06$ [yr$^{{-1}}$]",
    xy=(acceptable_as, pf_at(acceptable_as)),
    xytext=(93.2, 2.4e-5),
    arrowprops={"arrowstyle": "-", "color": "0.2", "linewidth": 0.8},
)

annualized_values = np.array([annualized_cost(As) for As in jcss_as_grid]) / 1e3
axes[1].plot(jcss_as_grid, annualized_values, color=line_color, linewidth=1.8)
axes[1].plot(optimal_as, annualized_cost(optimal_as) / 1e3, marker="o", markersize=9, markerfacecolor="none", markeredgecolor=opt_color, markeredgewidth=1.8)
axes[1].plot(acceptable_as, annualized_cost(acceptable_as) / 1e3, marker="s", markersize=9, markerfacecolor="none", markeredgecolor=acc_color, markeredgewidth=1.8)
axes[1].set_ylim(3.5, 5.5)
axes[1].set_ylabel(r"$C_y(A)$ [$10^3$ CHF yr$^{-1}$]")
axes[1].text(70.8, 5.15, "B)", fontsize=14, fontstyle="italic")
axes[1].annotate(
    rf"$C_y({optimal_as:.1f}\,mm^2)=4.26\,10^3$ [CHF yr$^{{-1}}$]",
    xy=(optimal_as, annualized_cost(optimal_as) / 1e3),
    xytext=(74.8, 4.63),
    arrowprops={"arrowstyle": "-", "color": "0.2", "linewidth": 0.8},
)
axes[1].annotate(
    rf"$C_y({acceptable_as:.0f}\,mm^2)=4.65\,10^3$ [CHF yr$^{{-1}}$]",
    xy=(acceptable_as, annualized_cost(acceptable_as) / 1e3),
    xytext=(88.8, 4.08),
    arrowprops={"arrowstyle": "-", "color": "0.2", "linewidth": 0.8},
)

lqi_values = np.array([paper_lqi_derivative(As) for As in jcss_as_grid]) * 1e5
axes[2].plot(jcss_as_grid, lqi_values, color=line_color, linewidth=1.8)
axes[2].axhline(0, color="0.2", linewidth=0.8)
axes[2].plot(optimal_as, paper_lqi_derivative(optimal_as) * 1e5, marker="o", markersize=9, markerfacecolor="none", markeredgecolor=opt_color, markeredgewidth=1.8)
axes[2].plot(acceptable_as, paper_lqi_derivative(acceptable_as) * 1e5, marker="s", markersize=9, markerfacecolor="none", markeredgecolor=acc_color, markeredgewidth=1.8)
axes[2].set_ylim(-5, 15)
axes[2].set_ylabel(r"LQI marginal term [$10^{-5}$]")
axes[2].text(70.8, 11.8, "C)", fontsize=14, fontstyle="italic")

objective_values = np.array([objective_at(As) for As in jcss_as_grid])
axes[3].semilogy(jcss_as_grid, objective_values, color=line_color, linewidth=1.8)
axes[3].plot(optimal_as, objective_at(optimal_as), marker="o", markersize=9, markerfacecolor="none", markeredgecolor=opt_color, markeredgewidth=1.8)
axes[3].plot(acceptable_as, objective_at(acceptable_as), marker="s", markersize=9, markerfacecolor="none", markeredgecolor=acc_color, markeredgewidth=1.8)
axes[3].set_ylim(1e4, 1.1e5)
axes[3].set_ylabel("objective function\nZ(A) [CHF]")
axes[3].set_xlabel(r"cross section [mm$^2$]")
axes[3].text(70.8, 7.0e4, "D)", fontsize=14, fontstyle="italic")
axes[3].annotate(
    rf"$Z_{{max}}({optimal_as:.1f}\,mm^2)=7.22\,10^4$ [CHF]",
    xy=(optimal_as, objective_at(optimal_as)),
    xytext=(81.5, 4.6e4),
    arrowprops={"arrowstyle": "-", "color": "0.2", "linewidth": 0.8},
)
axes[3].annotate(
    rf"$Z({acceptable_as:.0f}\,mm^2)=5.24\,10^4$ [CHF]",
    xy=(acceptable_as, objective_at(acceptable_as)),
    xytext=(89.5, 2.6e4),
    arrowprops={"arrowstyle": "-", "color": "0.2", "linewidth": 0.8},
)

fig.text(0.43, 0.035, rf"optimal alternative = {optimal_as:.1f} mm$^2$", ha="center")
fig.text(0.76, 0.012, rf"acceptable alternative = {acceptable_as:.0f} mm$^2$", ha="center")
fig.suptitle("JCSS LQI steel-bar example", y=0.995)
fig.tight_layout(rect=[0, 0.05, 1, 0.98])
